# Example: LTV State-Space Identification with `sid.ltv_disc`

This example demonstrates `sid.ltv_disc` (COSMIC algorithm) for identifying
time-varying discrete linear systems of the form:

$$x(k+1) = A(k)\,x(k) + B(k)\,u(k)$$

It also demonstrates `sid.ltv_disc_tune` for automatic regularisation tuning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sid

%matplotlib inline

## LTI System Recovery

First, verify that `sid.ltv_disc` correctly identifies a known LTI system
where A and B are constant.

In [ ]:
rng = np.random.default_rng(100)
p, q, N, L = 2, 1, 50, 10
A_true = np.array([[0.9, 0.1], [-0.1, 0.8]])
B_true = np.array([[1.0], [0.5]])
sigma = 0.02

# Generate L trajectories
X = np.zeros((N + 1, p, L))
U = rng.standard_normal((N, q, L))
for l in range(L):
    X[0, :, l] = rng.standard_normal(p)
    for k in range(N):
        X[k + 1, :, l] = (A_true @ X[k, :, l] + B_true @ U[k, :, l]
                          + sigma * rng.standard_normal(p))

result_lti = sid.ltv_disc(X, U, lambda_=1e5)

# a[:,:,k] should be approximately constant
A_mean = result_lti.a.mean(axis=2)
print('True A:\n', A_true)
print('Mean recovered A:\n', A_mean)
print(f'Recovery error: {np.linalg.norm(A_mean - A_true, "fro"):.4f}')

## LTV System: Time-Varying Pole

The (1,1) entry of A ramps from 0.5 to 0.9 over time.

In [ ]:
rng = np.random.default_rng(200)
N, L = 80, 15
a_ramp = np.linspace(0.5, 0.9, N)
A_tv = np.zeros((p, p, N))
for k in range(N):
    A_tv[:, :, k] = np.array([[a_ramp[k], 0.1], [-0.1, 0.8]])

X_tv = np.zeros((N + 1, p, L))
U_tv = rng.standard_normal((N, q, L))
for l in range(L):
    X_tv[0, :, l] = rng.standard_normal(p)
    for k in range(N):
        X_tv[k + 1, :, l] = (A_tv[:, :, k] @ X_tv[k, :, l]
                             + B_true @ U_tv[k, :, l]
                             + 0.05 * rng.standard_normal(p))

result_tv = sid.ltv_disc(X_tv, U_tv, lambda_=1e4)

# Plot recovered A(1,1,k) vs true ramp
fig, ax = plt.subplots()
ax.plot(range(1, N + 1), result_tv.a[0, 0, :], 'b', label='Recovered $A_{11}(k)$')
ax.plot(range(1, N + 1), a_ramp, 'k--', linewidth=1.5, label='True $A_{11}(k)$')
ax.set_xlabel('Time step k')
ax.set_ylabel('$A_{11}(k)$')
ax.set_title('LTV Identification: Time-Varying Pole Recovery')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Automatic Lambda Selection (L-curve)

With `lambda_='auto'`, `sid.ltv_disc` uses the L-curve method to find the
regularisation that best balances data fidelity and smoothness.

In [ ]:
result_auto = sid.ltv_disc(X_tv, U_tv, lambda_='auto')
print(f'Automatic lambda: {result_auto.lambda_[0]:.2e}')

fig, ax = plt.subplots()
ax.plot(range(1, N + 1), result_auto.a[0, 0, :], 'r', label=r'Auto $\lambda$')
ax.plot(range(1, N + 1), result_tv.a[0, 0, :], 'b', label=r'$\lambda = 10^4$')
ax.plot(range(1, N + 1), a_ramp, 'k--', linewidth=1.5, label='True')
ax.set_xlabel('Time step k')
ax.set_ylabel('$A_{11}(k)$')
ax.set_title('Manual vs Automatic Lambda')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Multi-Trajectory Benefit

More trajectories provide more information and reduce estimation error.

In [ ]:
rng = np.random.default_rng(300)
L_few, L_many = 3, 20

X_few = X_tv[:, :, :L_few]
U_few = U_tv[:, :, :L_few]

X_many = np.zeros((N + 1, p, L_many))
U_many = rng.standard_normal((N, q, L_many))
for l in range(L_many):
    X_many[0, :, l] = rng.standard_normal(p)
    for k in range(N):
        X_many[k + 1, :, l] = (A_tv[:, :, k] @ X_many[k, :, l]
                               + B_true @ U_many[k, :, l]
                               + 0.05 * rng.standard_normal(p))

r_few = sid.ltv_disc(X_few, U_few, lambda_=1e4)
r_many = sid.ltv_disc(X_many, U_many, lambda_=1e4)

fig, ax = plt.subplots()
ax.plot(range(1, N + 1), r_few.a[0, 0, :], 'r', label=f'L = {L_few}')
ax.plot(range(1, N + 1), r_many.a[0, 0, :], 'b', label=f'L = {L_many}')
ax.plot(range(1, N + 1), a_ramp, 'k--', linewidth=1.5, label='True')
ax.set_xlabel('Time step k')
ax.set_ylabel('$A_{11}(k)$')
ax.set_title('Effect of Number of Trajectories')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Validation-Based Lambda Tuning with `sid.ltv_disc_tune`

Split trajectories into training and validation sets, then search over
a grid of lambda values to minimise validation prediction error.

In [ ]:
rng = np.random.default_rng(400)
L_total, L_train = 20, 14

X_all = np.zeros((N + 1, p, L_total))
U_all = rng.standard_normal((N, q, L_total))
for l in range(L_total):
    X_all[0, :, l] = rng.standard_normal(p)
    for k in range(N):
        X_all[k + 1, :, l] = (A_tv[:, :, k] @ X_all[k, :, l]
                              + B_true @ U_all[k, :, l]
                              + 0.05 * rng.standard_normal(p))

X_train = X_all[:, :, :L_train]
U_train = U_all[:, :, :L_train]
X_val = X_all[:, :, L_train:]
U_val = U_all[:, :, L_train:]

grid_lam = np.logspace(0, 10, 30)
best_result, best_lambda, all_losses = sid.ltv_disc_tune(
    X_train, U_train, X_val, U_val,
    lambda_grid=grid_lam,
)

print(f'Validation-tuned lambda: {best_lambda:.2e}')

# Plot validation RMSE vs lambda
fig, ax = plt.subplots()
ax.semilogx(grid_lam, all_losses, 'b.-')
ax.semilogx(best_lambda, min(all_losses), 'ro', markersize=10, linewidth=2)
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel('Validation RMSE')
ax.set_title('Lambda Tuning: Validation Loss Curve')
ax.grid(True)
plt.tight_layout()
plt.show()

## Preconditioning for Numerical Stability

Block-diagonal preconditioning can improve conditioning of the solve.

In [ ]:
result_pre = sid.ltv_disc(X_tv, U_tv, lambda_=1e4, precondition=True)
print(f'Preconditioned: {result_pre.preconditioned}')

## Cost Decomposition

`result.cost` contains `[total, data_fidelity, regularisation]`.

In [ ]:
print('Cost decomposition:')
print(f'  Total:          {result_tv.cost[0]:.4f}')
print(f'  Data fidelity:  {result_tv.cost[1]:.4f}')
print(f'  Regularisation: {result_tv.cost[2]:.4f}')
print(f'  Check: {result_tv.cost[0] - result_tv.cost[1] - result_tv.cost[2]:.4e} (should be ~0)')

## Uncertainty Quantification

Enable Bayesian posterior uncertainty to get standard deviations for each
`A(k)` and `B(k)` entry. The noise covariance is estimated from residuals.

In [ ]:
result_unc = sid.ltv_disc(X_tv, U_tv, lambda_=1e4, uncertainty=True)

print('Uncertainty results:')
print(f'  Noise covariance estimated: {result_unc.noise_cov_estimated}')
print(f'  Noise variance (trace/p):   {result_unc.noise_variance:.6f}')
print(f'  Degrees of freedom:         {result_unc.degrees_of_freedom:.1f}')
print(f'  NoiseCov:\n{result_unc.noise_cov}')

# Plot A(1,1,k) with +/- 2 sigma uncertainty band
a11 = result_unc.a[0, 0, :]
a11_std = result_unc.a_std[0, 0, :]
kk = np.arange(1, N + 1)

fig, ax = plt.subplots()
ax.fill_between(kk, a11 - 2 * a11_std, a11 + 2 * a11_std,
                color=[0.8, 0.8, 1.0], label=r'$\pm 2\sigma$')
ax.plot(kk, a11, 'b', linewidth=1.5, label='Recovered $A_{11}(k)$')
ax.plot(kk, a_ramp, 'k--', linewidth=1.5, label='True $A_{11}(k)$')
ax.set_xlabel('Time step k')
ax.set_ylabel('$A_{11}(k)$')
ax.set_title('LTV Identification with Uncertainty Bands')
ax.legend(loc='lower right')
ax.grid(True)
plt.tight_layout()
plt.show()

## Frozen Transfer Function with `sid.ltv_disc_frozen`

Compute the instantaneous frequency response
$G(\omega, k) = (e^{j\omega}I - A(k))^{-1} B(k)$
at selected time steps. When the LTV result includes uncertainty,
`response_std` is propagated via first-order linearisation.

In [ ]:
k_steps = np.array([0, N // 2, N - 1])  # Python 0-indexed
frz = sid.ltv_disc_frozen(result_unc, time_steps=k_steps)

print('Frozen transfer function:')
print(f'  Method: {frz.method}')
print(f'  Response shape: {frz.response.shape}')
print(f'  ResponseStd available: {frz.response_std is not None}')

# Plot Bode magnitude at the three time steps
fig, ax = plt.subplots()
w = frz.frequency
colors = ['b', 'r', (0, 0.6, 0)]
for i, ks in enumerate(k_steps):
    G_k = frz.response[:, 0, 0, i]
    mag_dB = 20 * np.log10(np.abs(G_k))
    ax.semilogx(w, mag_dB, color=colors[i], linewidth=1.5,
                label=f'k = {ks + 1}')

# Add +/- 2 sigma band for the last time step
G_last = frz.response[:, 0, 0, -1]
G_std = frz.response_std[:, 0, 0, -1]
mag_upper = 20 * np.log10(np.abs(G_last) + 2 * G_std)
mag_lower = 20 * np.log10(np.maximum(np.abs(G_last) - 2 * G_std, np.finfo(float).eps))
ax.fill_between(w, mag_lower, mag_upper,
                color=[0.8, 1.0, 0.8], alpha=0.5)
# Re-plot the line on top
ax.semilogx(w, 20 * np.log10(np.abs(G_last)),
            color=colors[-1], linewidth=1.5)

ax.set_xlabel('Frequency (rad/sample)')
ax.set_ylabel('|G(w,k)| (dB)')
ax.set_title('Frozen Transfer Function at Selected Time Steps')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Frequency-Based Lambda Tuning (No Validation Data Needed)

`sid.ltv_disc_tune` with `method='frequency'` selects lambda by comparing
the COSMIC frozen transfer function against a non-parametric `sid.freq_map`
estimate. No validation data is required -- only training data.

In [ ]:
grid_freq = np.logspace(1, 8, 12)
best_result_freq, best_lambda_freq, info_freq = sid.ltv_disc_tune(
    X, U,
    method='frequency',
    lambda_grid=grid_freq,
    segment_length=20,
)

print(f'Frequency-tuned lambda: {best_lambda_freq:.2e} '
      f'(consistency: {info_freq["best_fraction"] * 100:.1f}%)')
print(f'Validation-tuned lambda: {best_lambda:.2e}')

# Plot consistency fraction vs lambda
fig, ax = plt.subplots()
ax.semilogx(info_freq["lambda_grid"], info_freq["fractions"] * 100, 'b.-', markersize=10)
ax.semilogx(best_lambda_freq, info_freq["best_fraction"] * 100, 'ro',
            markersize=12, linewidth=2)
ax.axhline(90, color='k', linestyle='--')
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel('Consistency fraction (%)')
ax.set_title('Frequency-Response Consistency Scoring')
ax.legend(['Consistency', r'Selected $\lambda$', '90% threshold'])
ax.grid(True)
plt.tight_layout()
plt.show()

## Model Validation with `sid.compare` and `sid.residual`

Compare COSMIC model predictions against the training data.

In [ ]:
comp = sid.compare(result_auto, X_tv, U_tv)
print('COSMIC model fit (per state component):')
for ch in range(p):
    print(f'  x_{ch + 1}: {comp["fit"][ch]:.1f}%')

# Residual diagnostics
resid = sid.residual(result_auto, X_tv, U_tv)
if resid['whiteness_pass']:
    print('Residual whiteness test: PASS')
else:
    print('Residual whiteness test: FAIL (model may need refinement)')